## BART

Extract the BART trials from the PsychoPy logs and summarize each round.

In [ ]:
import pandas as pd 
import openpyxl

Per-round maxPumps, size, earnings and timing.

In [ ]:
from pathlib import Path

# PsychoPy CSV exports. Visit 1 files are numbered 0nn, visit 2 files 1nn;
# only participants with both visits are kept.
DATA_DIR = Path('../../data/study1/behavior')

def by_number(pattern):
    return {int(f.name[:3]) % 100: f for f in sorted(DATA_DIR.glob(pattern))}

v1 = by_number('0[0-9][0-9]_*_brainpt_*.csv')
v2 = by_number('1[0-9][0-9]_*_brainpt_*.csv')
common = sorted(v1.keys() & v2.keys())

visit1_files = [v1[n] for n in common]
visit2_files = [v2[n] for n in common]

In [ ]:
def process_summary(filepath):
    df = pd.read_csv(filepath)
    cols = ['maxPumps', 'trial.started', 'trial.stopped', 'nPumps', 'size', 'earnings', 'popped']
    df = df[cols].dropna().reset_index(drop=True)

    rounds = []
    current_round = 1
    for i, row in df.iterrows():
        if row['nPumps'] == 1 and i != 0:
            current_round += 1
        rounds.append(current_round)
    df['round'] = rounds

    summary = df.groupby('round').agg({
        'nPumps': 'max',
        'maxPumps': 'max',
        'size': 'last',
        'earnings': 'last',
        'trial.started': 'first',
        'trial.stopped': 'last'
    }).reset_index()

    summary['round_duration'] = summary['trial.stopped'] - summary['trial.started']
    summary['average_duration'] = summary['round_duration'] / summary['nPumps']
    summary['risk_margin'] = summary['maxPumps'] - summary['nPumps']


    return summary.rename(columns={
        'nPumps': 'final_nPumps',
        'maxPumps': 'final_maxPumps',
        'size': 'final_size',
        'earnings': 'final_earnings',
        'trial.started': 'round_started',
        'trial.stopped': 'round_ended'
    })

In [ ]:
with pd.ExcelWriter("bart_summary.xlsx") as writer:
    for i in range(len(visit1_files)):
        v1 = process_summary(visit1_files[i]).add_prefix("visit1_")
        v2 = process_summary(visit2_files[i]).add_prefix("visit2_")
        merged = pd.concat([v1.reset_index(drop=True), v2.reset_index(drop=True)], axis=1)
        
        subject_id = visit1_files[i].name.split('_')[0] 
        sheet_name = f"sub{subject_id}"

        merged.to_excel(writer, index=False, sheet_name=sheet_name)

## Emotional Go/No-Go

Per participant: raw trials, summaries by trial type / emotion / colour, and error trials.

In [ ]:
def process_gonogo_summary(filepath):
    df = pd.read_csv(filepath)
    cols = [
        'Express', 'FaceE', 'ColourT', 'CorrectAnsP',
        'key_resp_12.keys', 'key_resp_12.corr', 'key_resp_12.rt'
    ]
    df = df[cols].dropna().copy().reset_index(drop=True)
    df['trial_type'] = df['CorrectAnsP'].apply(lambda x: 'No-Go' if x == 'none' else 'Go')

    summary_by_type = df.groupby('trial_type').agg({
        'key_resp_12.corr': 'mean',
        'key_resp_12.rt': ['mean', 'std', 'count']
    }).reset_index()
    summary_by_type.columns = ['trial_type', 'accuracy', 'rt_mean', 'rt_std', 'n_trials']

    summary_by_emotion = df.groupby('Express').agg({
        'key_resp_12.corr': 'mean',
        'key_resp_12.rt': 'mean'
    }).reset_index()
    summary_by_emotion.columns = ['emotion', 'accuracy', 'rt_mean']

    summary_by_colour = df.groupby('ColourT').agg({
        'key_resp_12.corr': 'mean',
        'key_resp_12.rt': 'mean'
    }).reset_index()
    summary_by_colour.columns = ['colour', 'accuracy', 'rt_mean']

    incorrect_trials = df[df['key_resp_12.corr'] == 0]

    return df, summary_by_type, summary_by_emotion, summary_by_colour, incorrect_trials

with pd.ExcelWriter("gonogo_all_subjects.xlsx") as writer:
    for filepath in visit1_files + visit2_files:
        subject_id = filepath.name.split('_')[0] 
        sheet_name = f"sub{subject_id}"
        raw, type_sum, emo_sum, col_sum, incorrect = process_gonogo_summary(filepath)

        raw.to_excel(writer, index=False, sheet_name=sheet_name, startrow=0)
        type_sum.to_excel(writer, index=False, sheet_name=sheet_name, startrow=len(raw)+3)
        emo_sum.to_excel(writer, index=False, sheet_name=sheet_name, startrow=len(raw)+len(type_sum)+6)
        col_sum.to_excel(writer, index=False, sheet_name=sheet_name, startrow=len(raw)+len(type_sum)+len(emo_sum)+9)
        incorrect.to_excel(writer, index=False, sheet_name=sheet_name, startrow=len(raw)+len(type_sum)+len(emo_sum)+len(col_sum)+12)